# NBA Shot Quality — EDA

Exploratory scratch space. Run the full pipeline first:

```bash
python scripts/run_pipeline.py --use-cache
```

Then load the cached parquet and the latest model artifact below to explore.

In [ ]:
from pathlib import Path

import pandas as pd

from nba_shot_quality.config import GAMES_CACHE_PATH, SHOTS_CACHE_DIR
from nba_shot_quality.features import engineer_features
from nba_shot_quality.model import load_latest

In [ ]:
shots = pd.concat([pd.read_parquet(p) for p in SHOTS_CACHE_DIR.glob('*.parquet')], ignore_index=True)
games = pd.read_parquet(GAMES_CACHE_PATH)
engineered = engineer_features(shots, games)
engineered.head()

In [ ]:
# Coverage by zone
engineered.groupby('SHOT_ZONE_BASIC')['SHOT_MADE_FLAG'].agg(['count', 'mean']).round(3)

In [ ]:
# Load the trained model and score the full dataset
model = load_latest()
from nba_shot_quality.features.engineer import ALL_FEATURE_COLS
engineered['xfg_pred'] = model.predict_proba(engineered[list(ALL_FEATURE_COLS)])[:, 1]
engineered[['PLAYER_NAME', 'SHOT_ZONE_BASIC', 'SHOT_DISTANCE', 'SHOT_MADE_FLAG', 'xfg_pred']].head(20)